In [1]:
# ============================================================
# SET UP
# ============================================================

# Add the project root directory to the Python search path
import sys, os
sys.path.append(os.path.abspath('..')) 

# Core libraries
import numpy as np
import pandas as pd

# Import all project-wide settings
from config import *

# Import evaluation metrics
from src.evaluation.metrics  import evaluate_all_horizons

# Benchmark Econometric Models
from src.models.gjr_garch    import GJRGARCH
from src.models.har_garch    import HARGARCH
from src.models.mf2_garch    import MF2GARCH

# Machine Learning Models
from src.models.xgboost_model import XGBoostModel
from src.models.mlp_model     import MLPModel
from src.models.hybrid_model  import HybridModel

In [2]:
# Load data with features created
df = pd.read_excel('../data/processed/rv_features_created.xlsx')
df['date'] = pd.to_datetime(df['date'])

print(f'Loaded {len(df):,} rows  |  stocks: {df["symbol"].unique().tolist()}')
print(f'Columns: {df.columns.tolist()}')

Loaded 16,698 rows  |  stocks: ['JNJ', 'JPM', 'MSFT', 'NVDA', 'PG', 'XOM']
Columns: ['date', 'symbol', 'asset_type', 'volume', 'trades', 'open_price', 'close_price', 'high_price', 'low_price', 'pr', 'gkr', 'rr5', 'rv1', 'rv5', 'rv5_ss', 'bv1', 'bv5', 'bv5_ss', 'rsp1', 'rsn1', 'rsp5', 'rsn5', 'rsp5_ss', 'rsn5_ss', 'medrv1', 'medrv5', 'medrv5_ss', 'minrv1', 'minrv5', 'minrv5_ss', 'rk', 'rq1', 'rq5', 'rq5_ss', 'index', 'ret', 'ret_neg', 'rv', 'rv_lag_5', 'rv_lag_22', 'rv_roll_5', 'rv_roll_22', 'log_rv', 'log_rv_d', 'log_rv_w', 'log_rv_m', 'rv_ratio', 'target_rv_1', 'target_rv_5', 'target_rv_22']


In [3]:
# ============================================================
# ROLLING GJR-GARCH(1,1)
# ============================================================
'''
Purpose
-------
Generate rolling OOS RV forecasts using a GJR-GARCH(1,1) model.
The procedure mimics real-time forecasting:

    1. Estimate model using historical data
    2. Produce forecasts
    3. Move forward one day
    4. Update estimation sample
    5. Re-estimate periodically

This rolling-window framework avoids look-ahead bias and
provides a realistic assessment of forecasting performance.

Forecast Horizons
-----------------

h = 1   trading day
h = 5   trading days
h = 22  trading days

Re-estimation Strategy
----------------------

The model is re-estimated every:

    refit_frequency = 5 days

rather than every day.This substantially reduces 
computational cost while preserving forecast accuracy.
'''

def rolling_gjr_garch(
    train_returns,
    test_returns,
    window_size=rolling_window,
    refit_frequency=refit_frequency,
    horizons=forecast_horizons,
):
    # Initial rolling sample
    rolling_ret = train_returns.copy()

    # Max forecast horizon up to 22
    max_h = max(horizons)

    # Storage for forecasts
    forecasts = {h: [] for h in horizons}

    # Storage for estimated parameters
    params_log = {}

    # Storage for forecast dates
    dates = []

    # Placeholder for fitted model
    fitted = None

    # OOS forecasting loop
    for t in range(len(test_returns)):

        # Current forecast date
        date = test_returns.index[t]

        # Roll the window forward to INCLUDE the current day's return (known at the
        # forecast origin) before forecasting RV_{t+1}. Doing this at the top of the
        # loop aligns the GARCH information set with the HAR/ML models, which
        # condition on realized variance through day t.
        rolling_ret = pd.concat([rolling_ret, test_returns.iloc[t:t+1]]).iloc[-window_size:]

        # Periodic model re-estimation
        if t % refit_frequency == 0:

            # Estimate GJR-GARCH(1,1) using the rolling window
            fitted = GJRGARCH().fit(rolling_ret)

            # Save parameter estimates
            params_log[date] = fitted.parameters()

        # Forecast the full multi-step variance path [h=1, ..., h=22] from the
        # current window end. forecast_fixed re-filters the GJR recursion with the
        # parameters held fixed, so the forecast origin advances daily between the
        # 5-day re-estimations (matching the HAR/ML models) instead of staying
        # pinned at the last refit date.
        path = fitted.forecast_fixed(rolling_ret, horizon=max_h)
        
        # Convert forecast path into avg variance forecast over horizon h
        for h in horizons:

            # Returns were scaled to percentage, since ret is in % form but 
            # GARCH variance is in %², so divide by 100² to recover var in decimal form
            forecasts[h].append(float(path.iloc[:h].mean()) / 1e4)

        # Store forecast date
        dates.append(date)

    # Create forecast storage table
    forecast_df = pd.DataFrame({f'gjr_h{h}': forecasts[h] for h in horizons}, index=dates)

    # Stores all parameter estimates obtained at each refit date.
    params_df = pd.DataFrame(params_log).T

    # Label index for export and plotting
    params_df.index.name = 'date'

    # Return these two tables
    return forecast_df, params_df

In [ ]:
# ============================================================
# ROLLING MF2-GARCH
# ============================================================
'''
Purpose
-------
Generate rolling OOS RV forecasts using the MF2-GARCH model.
The MF2-GARCH model decomposes conditional variance into:

      σ²_t = h_t × τ_t

where:

      h_t   = short-run volatility component
      τ_t   = long-run volatility component

This decomposition allows the model to separately capture:

  • high-frequency volatility fluctuations
  • persistent volatility regimes
'''

def rolling_mf2_garch(
    train_returns, 
    test_returns,
    m=mf2_m,
    window_size=rolling_window,
    refit_frequency=refit_frequency,
    horizons=forecast_horizons,
):  
    
    rolling_ret = train_returns.copy()
    max_h = max(horizons)
    forecasts = {h: [] for h in horizons}
    params_log, dates = {}, []

    fitted = None
    last_x = None  # warm-start vector; updated after every refit

    # Rolling OOS forecasting loop
    for t in range(len(test_returns)):
        date = test_returns.index[t]

        # Include the current day's return in the information set before forecasting
        # RV_{t+1}, matching the HAR/ML models (see rolling_gjr_garch for rationale).
        rolling_ret = pd.concat([rolling_ret, test_returns.iloc[t:t+1]]).iloc[-window_size:]

        if t % refit_frequency == 0:

            # Estimate MF2-GARCH; pass previous estimates as warm start so
            # the optimizer only needs 1-2 starting points instead of many.
            fitted = MF2GARCH.fit(rolling_ret, m=m, warm_start=last_x)

            # Cache parameter vector for next refit
            last_x = np.array([
                fitted.alpha, fitted.beta, fitted.gamma,
                fitted.lambda0, fitted.lambda1, fitted.lambda2,
            ])

            params_log[date] = fitted.parameters()

        # Advance the forecast origin DAILY: re-filter the MF2 recursion over the
        # current window (ending at day t) with the parameters held fixed, exactly
        # as GJR-GARCH does via forecast_fixed(). Re-estimation still happens only
        # every refit_frequency days, but the forecast conditions on the latest
        # returns each day, keeping the two econometric benchmarks on an identical
        # information set between re-estimations.
        fitted._returns = rolling_ret.values.astype(float)
        path = fitted.forecast(horizon=max_h)

        for h in horizons:
            forecasts[h].append(float(path[:h].mean()) / 1e4)
        dates.append(date)

    forecast_df = pd.DataFrame({f'mf2_h{h}': forecasts[h] for h in horizons}, index=dates)
    params_df = pd.DataFrame(params_log).T
    params_df.index.name = 'date'

    return forecast_df, params_df

In [ ]:
# ============================================================
# ROLLING HAR-GARCH
# ============================================================
'''
Purpose
-------
Generate rolling OOS realized volatility forecasts using 
the HAR-GARCH model. The HAR-GARCH specification combines:

    1. HAR mean equation
    2. GARCH(1,1) volatility dynamics

The HAR component captures long-memory behavior in
realized volatility through daily, weekly, and monthly
volatility measures.
'''

def rolling_har_garch(
    train_df,
    test_df,
    window_size=rolling_window,
    refit_frequency=refit_frequency,
    horizons=forecast_horizons,
):
    rolling_df = train_df.copy()
    forecasts  = {h: [] for h in horizons}
    params_log, dates = {}, []
    fitted = None

    # Rolling OOS forecasting loop
    for t in range(len(test_df)):
        date = test_df.index[t]

        if t % refit_frequency == 0:

            # Estimate HAR-GARCH using the most recent rolling window
            fitted = HARGARCH().fit(rolling_df.iloc[-window_size:])

            # Store estimated coefficients(HAR coeff + GARCH parameters)
            params_log[date] = fitted.parameters()

        pred = fitted.forecast(test_df.iloc[t])
        
        for h in horizons:
            forecasts[h].append(pred[h])
        dates.append(date)
        rolling_df = pd.concat([rolling_df, test_df.iloc[t:t+1]]).iloc[-window_size:]

    forecast_df = pd.DataFrame({f'har_h{h}': forecasts[h] for h in horizons}, index=dates)
    params_df = pd.DataFrame(params_log).T
    params_df.index.name = 'date'

    return forecast_df, params_df

In [ ]:
# ============================================================
# ROLLING XGBOOST
# ============================================================
'''
Purpose
-------
Generate rolling OOS RV forecasts using the XGBoost ML model.

Unlike econometric volatility models, XGBoost does not model
conditional variance dynamics directly. Instead, it learns a
nonlinear relationship between lagged realized-volatility
features and future realized volatility.
'''

def rolling_xgboost(
    train_df,
    test_df,
    features=xgb_features,
    window_size=rolling_window,
    refit_frequency=refit_frequency,
    horizons=forecast_horizons,
):
    rolling_df = train_df.copy()
    forecasts  = {h: [] for h in horizons}
    dates = []
    fitted = None

    # Rolling OOS forecasting loop
    for t in range(len(test_df)):

        # Current forecast date used as the index for forecast storage
        date = test_df.index[t]

        # Periodic model retraining
        if t % refit_frequency == 0:

            # Extract the most recent rolling estimation window.
            window  = rolling_df.iloc[-window_size:]

            # Initialize XGBoost forecasting model
            fitted  = XGBoostModel()

            # Train XGBoost using the selected RV features
            fitted.fit(window[features], window)

        pred = fitted.forecast(test_df.iloc[t:t+1][features])
        
        for h in horizons:
            forecasts[h].append(pred[h])
        dates.append(date)
        rolling_df = pd.concat([rolling_df, test_df.iloc[t:t+1]]).iloc[-window_size:]

    return pd.DataFrame({f'xgb_h{h}': forecasts[h] for h in horizons}, index=dates)

In [ ]:
# ============================================================
# ROLLING MLP
# ============================================================
'''
Purpose
-------
Generate rolling out-of-sample realized volatility forecasts
using a Multi-Layer Perceptron (MLP) neural network.

The MLP is a feed-forward artificial neural network capable
of capturing nonlinear relationships between lagged realized
volatility features and future realized volatility.

Unlike econometric volatility models, the MLP does not impose
a parametric structure on volatility dynamics. Instead, it
learns nonlinear mappings directly from the data.
'''
def rolling_mlp(
    train_df,
    test_df,
    features=mlp_features,
    window_size=rolling_window,
    refit_frequency=refit_frequency,
    horizons=forecast_horizons,
):
    rolling_df = train_df.copy()
    forecasts  = {h: [] for h in horizons}
    dates = []
    fitted = None

    for t in range(len(test_df)):
        date = test_df.index[t]

        if t % refit_frequency == 0:
            fitted = MLPModel()
            fitted.fit(rolling_df.iloc[-window_size:], features)

        pred = fitted.forecast(test_df.iloc[t:t+1][features])
        
        for h in horizons:
            forecasts[h].append(pred[h])
        dates.append(date)
        rolling_df = pd.concat([rolling_df, test_df.iloc[t:t+1]]).iloc[-window_size:]

    return pd.DataFrame({f'mlp_h{h}': forecasts[h] for h in horizons}, index=dates)

In [ ]:
# ============================================================
# ROLLING HYBRID  (MF2-GARCH state features + XGBoost)
# ============================================================
'''
Model:
  Hybrid = MF2-GARCH + XGBoost

Forecasting Framework
---------------------
The procedure combines econometric volatility-state information
from MF2-GARCH with the nonlinear predictive power of XGBoost.

At each refit date:

  1. Estimate MF2-GARCH using the most recent rolling window
     of daily returns.

  2. Extract the short-run volatility component (h_t)
     and long-run volatility component (tau_t).

  3. Augment the standard realized-volatility feature set
     with MF2-GARCH state variables:

         mf2_h
         mf2_tau

  4. Train three separate XGBoost models:

         h = 1 day
         h = 5 days
         h = 22 days

Between refits:

  • Model parameters remain fixed.
  • The MF2-GARCH regime features (mf2_h, mf2_tau) are held at their last
    in-sample values. Re-filtering them daily was tested and moves QLIKE only
    marginally (see HybridModel.forecast), so the frozen state is kept.

The 5-day rolling re-estimation and the daily-updated log-RV features mirror
the framework used for the other models, and the procedure avoids look-ahead
bias.
'''

def rolling_hybrid(
    train_df,
    test_df,
    features=xgb_features,
    m=mf2_m,
    window_size=rolling_window,
    refit_frequency=refit_frequency,
    horizons=forecast_horizons,
):
    rolling_df = train_df.copy()
    forecasts  = {h: [] for h in horizons}
    dates = []
    fitted = None

    for t in range(len(test_df)):
        date = test_df.index[t]

        # Periodic neural network retraining, and between retraining dates, 
        # forecasts are generated using the most recently fitted network parameters
        if t % refit_frequency == 0:
            fitted = HybridModel(base_features=features, m=m)
            fitted.fit(rolling_df.iloc[-window_size:])

        pred = fitted.forecast(test_df.iloc[t])

        for h in horizons:
            forecasts[h].append(pred[h])
        dates.append(date)
        rolling_df = pd.concat([rolling_df, test_df.iloc[t:t+1]]).iloc[-window_size:]

    return pd.DataFrame({f'hybrid_h{h}': forecasts[h] for h in horizons}, index=dates)

In [9]:
# ============================================================
# RUN  —  GJR-GARCH
# ============================================================

for stock in stocks:
    
    # Display currently processed stock
    print(f'[GJR-GARCH] {stock} ...', end=' ')

    # Extract stock-specific dataset
    sdf = df[df['symbol'] == stock].copy()

    # Generate rolling forecasts
    fcst, params = rolling_gjr_garch(
        sdf[sdf['date'] < split_date]['ret'],   # In-sample estimation period
        sdf[sdf['date'] >= split_date]['ret'],  # Out-of-sample evaluation period
    )

    # Extract out-of-sample observations
    test_df = sdf[sdf['date'] >= split_date]

    # Combine forecasts with actual values, align by date
    eval_df = test_df.merge(fcst, left_index=True, right_index=True)

    # Forecast evaluation
    metrics = evaluate_all_horizons(eval_df, 'gjr')

    # Save forecast series
    fcst.to_csv(f'{FORECAST_DIR}/{stock}_GJR.csv')

    # Save parameter history with calendar-date index
    params.index = test_df.loc[params.index, 'date'].values
    params.index.name = 'date'
    params.to_csv(f'{TABLE_DIR}/{stock}_GJR_params.csv')

    # Save evaluation metrics
    metrics.to_csv(f'{METRICS_DIR}/{stock}_GJR_metrics.csv', index=False)

    # Display completion message
    print('done')
    print(metrics.to_string(index=False))
    print()

[GJR-GARCH] MSFT ... done
Model  Horizon          MSE    QLIKE
  gjr        1 1.608142e-07 0.395710
  gjr        5 5.123388e-08 0.287603
  gjr       22 1.959700e-08 0.179362

[GJR-GARCH] JNJ ... done
Model  Horizon          MSE    QLIKE
  gjr        1 1.061546e-08 0.211141
  gjr        5 4.308731e-09 0.123594
  gjr       22 1.986585e-09 0.075323

[GJR-GARCH] PG ... done
Model  Horizon          MSE    QLIKE
  gjr        1 8.072037e-09 0.179819
  gjr        5 4.131559e-09 0.108242
  gjr       22 1.893507e-09 0.070065

[GJR-GARCH] XOM ... done
Model  Horizon          MSE    QLIKE
  gjr        1 8.272894e-08 0.222979
  gjr        5 3.204546e-08 0.160218
  gjr       22 1.926754e-08 0.122776

[GJR-GARCH] JPM ... done
Model  Horizon          MSE    QLIKE
  gjr        1 6.292774e-08 0.227491
  gjr        5 3.420370e-08 0.174084
  gjr       22 1.354102e-08 0.126757

[GJR-GARCH] NVDA ... done
Model  Horizon          MSE    QLIKE
  gjr        1 3.804398e-07 0.205544
  gjr        5 2.086314e-07 0.

In [4]:
# ============================================================
# RUN  —  HAR-GARCH
# ============================================================

for stock in stocks:

    print(f'[HAR-GARCH] {stock} ...', end=' ')
    sdf      = df[df['symbol'] == stock].copy()
    train_df = sdf[sdf['date'] < split_date]
    test_df  = sdf[sdf['date'] >= split_date]

    fcst, params = rolling_har_garch(train_df, test_df)
    eval_df = test_df.merge(fcst, left_index=True, right_index=True)
    metrics = evaluate_all_horizons(eval_df, 'har')

    fcst.to_csv(f'{FORECAST_DIR}/{stock}_HAR.csv')
    params.index = test_df.loc[params.index, 'date'].values
    params.index.name = 'date'
    params.to_csv(f'{TABLE_DIR}/{stock}_HAR_params.csv')
    metrics.to_csv(f'{METRICS_DIR}/{stock}_HAR_metrics.csv', index=False)

    print('done')
    print(metrics.to_string(index=False))
    print()


[HAR-GARCH] MSFT ... done
Model  Horizon          MSE    QLIKE
  har        1 1.686635e-07 0.354242
  har        5 5.179924e-08 0.259278
  har       22 1.770942e-08 0.157513

[HAR-GARCH] JNJ ... done
Model  Horizon          MSE    QLIKE
  har        1 9.851833e-09 0.177692
  har        5 4.024986e-09 0.104931
  har       22 1.939728e-09 0.063747

[HAR-GARCH] PG ... done
Model  Horizon          MSE    QLIKE
  har        1 6.742966e-09 0.146829
  har        5 3.388619e-09 0.091001
  har       22 1.817760e-09 0.060335

[HAR-GARCH] XOM ... done
Model  Horizon          MSE    QLIKE
  har        1 7.732837e-08 0.168601
  har        5 2.886902e-08 0.128473
  har       22 1.607223e-08 0.095383

[HAR-GARCH] JPM ... done
Model  Horizon          MSE    QLIKE
  har        1 5.270029e-08 0.178875
  har        5 3.035499e-08 0.135346
  har       22 1.354898e-08 0.099868

[HAR-GARCH] NVDA ... done
Model  Horizon          MSE    QLIKE
  har        1 3.043183e-07 0.153618
  har        5 1.701583e-07 0.

In [11]:
# ============================================================
# RUN  —  MF2-GARCH
# ============================================================

for stock in stocks:

    print(f'[MF2-GARCH] {stock} ...', end=' ')
    
    sdf = df[df['symbol'] == stock].copy()

    fcst, params = rolling_mf2_garch(
        sdf[sdf['date'] < split_date]['ret'],
        sdf[sdf['date'] >= split_date]['ret'],
    )
    test_df = sdf[sdf['date'] >= split_date]
    eval_df = test_df.merge(fcst, left_index=True, right_index=True)
    metrics = evaluate_all_horizons(eval_df, 'mf2')

    fcst.to_csv(f'{FORECAST_DIR}/{stock}_MF2.csv')
    params.index = test_df.loc[params.index, 'date'].values
    params.index.name = 'date'
    params.to_csv(f'{TABLE_DIR}/{stock}_MF2_params.csv')
    metrics.to_csv(f'{METRICS_DIR}/{stock}_MF2_metrics.csv', index=False)
    
    print('done')
    print(metrics.to_string(index=False))
    print()

[MF2-GARCH] MSFT ... done
Model  Horizon          MSE    QLIKE
  mf2        1 1.631926e-07 0.404413
  mf2        5 5.400343e-08 0.298483
  mf2       22 2.114838e-08 0.186133

[MF2-GARCH] JNJ ... done
Model  Horizon          MSE    QLIKE
  mf2        1 1.088015e-08 0.219690
  mf2        5 4.394679e-09 0.127919
  mf2       22 1.850902e-09 0.073153

[MF2-GARCH] PG ... done
Model  Horizon          MSE    QLIKE
  mf2        1 8.412709e-09 0.181070
  mf2        5 4.416367e-09 0.109273
  mf2       22 2.026652e-09 0.072427

[MF2-GARCH] XOM ... done
Model  Horizon          MSE    QLIKE
  mf2        1 8.281502e-08 0.238367
  mf2        5 3.229147e-08 0.176640
  mf2       22 2.085256e-08 0.149315

[MF2-GARCH] JPM ... done
Model  Horizon          MSE    QLIKE
  mf2        1 6.498875e-08 0.232125
  mf2        5 3.461500e-08 0.173476
  mf2       22 1.336422e-08 0.119481

[MF2-GARCH] NVDA ... done
Model  Horizon          MSE    QLIKE
  mf2        1 3.631144e-07 0.204389
  mf2        5 2.008405e-07 0.

In [12]:
# ============================================================
# RUN  —  XGBOOST
# ============================================================

for stock in stocks:

    print(f'[XGBoost] {stock} ...', end=' ')

    sdf      = df[df['symbol'] == stock].copy()
    train_df = sdf[sdf['date'] < split_date]
    test_df  = sdf[sdf['date'] >= split_date]
    
    fcst    = rolling_xgboost(train_df, test_df)
    eval_df = test_df.merge(fcst, left_index=True, right_index=True)
    metrics = evaluate_all_horizons(eval_df, 'xgb')

    fcst.to_csv(f'{FORECAST_DIR}/{stock}_XGB.csv')
    metrics.to_csv(f'{METRICS_DIR}/{stock}_XGB_metrics.csv', index=False)
    
    print('done')
    print(metrics.to_string(index=False))
    print()

[XGBoost] MSFT ... done
Model  Horizon          MSE    QLIKE
  xgb        1 1.576805e-07 0.345364
  xgb        5 4.864000e-08 0.241582
  xgb       22 2.036680e-08 0.144726

[XGBoost] JNJ ... done
Model  Horizon          MSE    QLIKE
  xgb        1 1.110327e-08 0.215134
  xgb        5 4.440700e-09 0.124884
  xgb       22 2.557341e-09 0.081654

[XGBoost] PG ... done
Model  Horizon          MSE    QLIKE
  xgb        1 1.027483e-08 0.202275
  xgb        5 4.227323e-09 0.135233
  xgb       22 3.979659e-09 0.103623

[XGBoost] XOM ... done
Model  Horizon          MSE    QLIKE
  xgb        1 9.445521e-08 0.203744
  xgb        5 6.309138e-08 0.157074
  xgb       22 5.745481e-08 0.148985

[XGBoost] JPM ... done
Model  Horizon          MSE    QLIKE
  xgb        1 6.378154e-08 0.219411
  xgb        5 3.689911e-08 0.180066
  xgb       22 4.172993e-08 0.159281

[XGBoost] NVDA ... done
Model  Horizon          MSE    QLIKE
  xgb        1 3.373307e-07 0.163322
  xgb        5 1.839274e-07 0.110953
  xgb

In [4]:
# ============================================================
# RUN  —  MLP
# ============================================================

for stock in stocks:

    print(f'[MLP] {stock} ...', end=' ')
    
    sdf      = df[df['symbol'] == stock].copy()
    train_df = sdf[sdf['date'] < split_date]
    test_df  = sdf[sdf['date'] >= split_date]

    fcst    = rolling_mlp(train_df, test_df)
    eval_df = test_df.merge(fcst, left_index=True, right_index=True)
    metrics = evaluate_all_horizons(eval_df, 'mlp')

    fcst.to_csv(f'{FORECAST_DIR}/{stock}_MLP.csv')
    metrics.to_csv(f'{METRICS_DIR}/{stock}_MLP_metrics.csv', index=False)
    
    print('done')
    print(metrics.to_string(index=False))
    print()

[MLP] MSFT ... done
Model  Horizon          MSE    QLIKE
  mlp        1 1.557206e-07 0.371821
  mlp        5 4.762152e-08 0.257639
  mlp       22 1.606641e-08 0.154792

[MLP] JNJ ... done
Model  Horizon          MSE    QLIKE
  mlp        1 1.018043e-08 0.186037
  mlp        5 4.091072e-09 0.108031
  mlp       22 1.865814e-09 0.065878

[MLP] PG ... done
Model  Horizon          MSE    QLIKE
  mlp        1 6.970054e-09 0.149870
  mlp        5 3.427696e-09 0.090274
  mlp       22 1.680383e-09 0.058736

[MLP] XOM ... done
Model  Horizon          MSE    QLIKE
  mlp        1 9.534924e-08 0.173415
  mlp        5 3.727629e-08 0.129535
  mlp       22 1.204190e-06 0.094765

[MLP] JPM ... done
Model  Horizon          MSE    QLIKE
  mlp        1 5.739403e-08 0.183533
  mlp        5 3.118301e-08 0.135790
  mlp       22 1.223493e-08 0.095842

[MLP] NVDA ... done
Model  Horizon          MSE    QLIKE
  mlp        1 3.223255e-07 0.158831
  mlp        5 1.682723e-07 0.104297
  mlp       22 8.465831e-08 0

In [14]:
# ============================================================
# RUN  —  HYBRID  (MF2-GARCH + XGBoost)
# ============================================================

for stock in stocks:

    print(f'[Hybrid] {stock} ...', end=' ')
    
    sdf      = df[df['symbol'] == stock].copy()
    train_df = sdf[sdf['date'] < split_date]
    test_df  = sdf[sdf['date'] >= split_date]

    fcst    = rolling_hybrid(train_df, test_df)
    eval_df = test_df.merge(fcst, left_index=True, right_index=True)
    metrics = evaluate_all_horizons(eval_df, 'hybrid')

    fcst.to_csv(f'{FORECAST_DIR}/{stock}_Hybrid.csv')
    metrics.to_csv(f'{METRICS_DIR}/{stock}_Hybrid_metrics.csv', index=False)
    
    print('done')
    print(metrics.to_string(index=False))
    print()

[Hybrid] MSFT ... done
 Model  Horizon          MSE    QLIKE
hybrid        1 1.600429e-07 0.354866
hybrid        5 4.945640e-08 0.242385
hybrid       22 2.079314e-08 0.141360

[Hybrid] JNJ ... done
 Model  Horizon          MSE    QLIKE
hybrid        1 1.105869e-08 0.214296
hybrid        5 4.380300e-09 0.123030
hybrid       22 1.930950e-09 0.069450

[Hybrid] PG ... done
 Model  Horizon          MSE    QLIKE
hybrid        1 1.050664e-08 0.200756
hybrid        5 4.290355e-09 0.136276
hybrid       22 5.088221e-09 0.102339

[Hybrid] XOM ... done
 Model  Horizon          MSE    QLIKE
hybrid        1 1.143977e-07 0.210147
hybrid        5 5.540072e-08 0.152354
hybrid       22 2.864055e-08 0.114255

[Hybrid] JPM ... done
 Model  Horizon          MSE    QLIKE
hybrid        1 6.322574e-08 0.222178
hybrid        5 3.669650e-08 0.180547
hybrid       22 2.764237e-08 0.152037

[Hybrid] NVDA ... done
 Model  Horizon          MSE    QLIKE
hybrid        1 3.491011e-07 0.169248
hybrid        5 1.770669e-